In [ ]:
# input
zinc_hct = "./data/cysteine_data_hct_all.csv"
zinc_hepatocyte = "./data/cysteine_data_hepatocyte_all.csv"
zinc_prostate = "./data/cysteine_data_prostate_all.csv"
human_high_conf_pred = "./tmp/human_high_conf_pred.tsv"
high_conf_pred_interface = "../../homomer/data/pred_homomer_interface.tsv"

In [2]:
import pandas as pd

def read_data(
    file: str,
    source: str,
):
    df = pd.read_csv(file)
    df = df.dropna()
    df = df[df.apply(lambda row: row['Significance'] != "ns", axis=1)]
    df['source'] = source
        
    return df

files_sources = [
    (zinc_hct, "hct"),
    (zinc_hepatocyte, "hepatocyte"),
    (zinc_prostate, "prostate")
]

dfs = []
for (f, s) in files_sources:
    dfs.append(read_data(f, s))
df = pd.concat(dfs)

In [3]:
def filter_for_tpen(row):
    if row['source'] == "prostate":
        return row['Ratio'] > 1.5
    else:
        return row['p_adj'] < 0.05 and row['Ratio'] > 1.5
    
def filter_for_edta(row):
    if row['source'] == "prostate":
        return row['Ratio'] > 1.5
    else:
        return row['p_adj'] < 0.05 and row['Ratio'] > 1.5
    
def filter_for_zncl2(row):
    if row['source'] == "prostate":
        return row['Ratio'] < 0.8
    else:
        return row['p_adj'] < 0.05 and row['Ratio'] < 0.8
    
df_tpen = df[df["Condition"] == "TPEN"]
df_tpen = df_tpen[df_tpen.apply(lambda row: filter_for_tpen(row), axis=1)]

df_zncl2 = df[df["Condition"].map(lambda x: x in {'Zinc', "ZnCl2"})]
df_zncl2 = df_zncl2[df_zncl2.apply(lambda row: filter_for_zncl2(row), axis=1)]

df_edta = df[df["Condition"].map(lambda x: x in {'EDTA_1mM', 'EDTA_5mM'})]
df_edta = df_edta[df_edta.apply(lambda row: filter_for_edta(row), axis=1)]

In [4]:
def get_residues(df: pd.DataFrame):
    residues = set()
    for _, row in df.iterrows():
        seq_id = row['Uniprot']
        sites = row['Cysteine_Site'].split("_")[-1].split(";")
        for s in sites:
            residues.add((seq_id, int(s)))
    return residues

# NOTE: these statistics are not consistent with the result from paper
tpen_res = get_residues(df_tpen) # it's 4830 in the paper
tpen_pro = set(df_tpen['Uniprot'])
len(tpen_res)
len(tpen_pro)

edta_res = get_residues(df_edta)
edta_pro = set(df_edta['Uniprot'])
len(edta_res)
len(edta_pro)

zncl2_res = get_residues(df_zncl2) # 1742 in the paper
zncl2_pro = set(df_zncl2['Uniprot'])
len(zncl2_res)
len(zncl2_pro)

4545

1670

4275

1305

1651

697

## compare with high conf pred

In [5]:
df = pd.read_table(human_high_conf_pred)
df['seq_id'] = df['seq_id'].map(lambda x: x.split("-")[1])

pred_pro = set(df['seq_id'])
len(pred_pro)

4513

In [6]:
constitutive_pro = tpen_pro | edta_pro
len(constitutive_pro)
len(pred_pro & constitutive_pro)

induced_pro = zncl2_pro
len(zncl2_pro)
len(pred_pro & induced_pro)

2109

1149

697

298

## compare with high conf pred (interface)

In [7]:
df = pd.read_table(high_conf_pred_interface)
df['seq_id'] = df['seq_id'].map(lambda x: x.split("-")[1])

pred_interface_pro = set(df['seq_id'])
len(pred_interface_pro)

386

In [10]:
len(pred_interface_pro & constitutive_pro)
len(pred_interface_pro & induced_pro)

print(pred_interface_pro & constitutive_pro)

38

10

{'P67870', 'Q8IUQ4', 'Q9Y4E5', 'P62072', 'Q9BU19', 'P00326', 'Q9NZ45', 'Q9H116', 'P17544', 'Q9BQG2', 'P53384', 'O43167', 'Q9HBE1', 'O43681', 'Q8N5K1', 'P28698', 'Q9NYP9', 'Q96GC6', 'Q03393', 'P15336', 'P11766', 'O15156', 'P07327', 'P10074', 'P00390', 'Q8IWY8', 'P02794', 'Q9UFB7', 'Q9Y5L4', 'Q7Z6V5', 'P15407', 'P00325', 'Q8TF39', 'P17535', 'Q9H4P4', 'Q9UH99', 'P33316', 'Q04760'}
